In [ ]:
from notebooks.data.OligoAI.assign_canonical_gene import process_and_assign_genes_bulk

df, ambiguous_genes, gene_stats = process_and_assign_genes_bulk(threads=16)

Extracted 155175 unique ASO sequences.
Running bulk alignment and annotation...
Loading GTF into RAM... (This takes ~45s but only happens once)
Inferring missing introns from transcript coordinates...
Intersecting 2143716 hits against the genome...


In [3]:
# If the most popular candidate agrees with the original candidate we can safely continue.
filtered_df = gene_stats[gene_stats['original_target_gene'] != gene_stats['most_popular_canonical']]

In [4]:
import ast
import pandas as pd

# 1. Safely parse the dictionary column
def parse_dict(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return {}
    return val if isinstance(val, dict) else {}

filtered_df = filtered_df.copy()
filtered_df['all_hit_frequencies'] = filtered_df['all_hit_frequencies'].apply(parse_dict)

# 2. Extract the hit count for the original target gene
def get_original_count(row):
    original = row['original_target_gene']
    freq_dict = row['all_hit_frequencies']
    return freq_dict.get(original, 0)

filtered_df['original_hit_count'] = filtered_df.apply(get_original_count, axis=1)

# 3. Use a 50% threshold to ensure we don't keep noise (like NAIP's 3 hits)
threshold = filtered_df['total_asos_tested'] * 0.5
present_originals = filtered_df[filtered_df['original_hit_count'] >= threshold]
missing_originals = filtered_df[filtered_df['original_hit_count'] < threshold]

# 4. Print the MISSING ORIGINALS (Now including exact original hits and total ASOs)
print(f"--- MISSING ORIGINALS ({len(missing_originals)} groups) ---")
print("These did not map to the original name at all (or missed the 50% threshold):\n")
missing_cols = [
    'original_target_gene',
    'original_hit_count',
    'total_asos_tested',
    'most_popular_canonical',
    'popular_hit_count',
    'match_rate'
]
print(missing_originals[missing_cols].to_string(index=False))

# 5. Print the RECOVERABLE ORIGINALS (Now including total ASOs tested for context)
print(f"\n--- RECOVERABLE ORIGINALS ({len(present_originals)} groups) ---")
print("These exist in the frequencies and can be reverted to the original name:\n")
recoverable_cols = [
    'original_target_gene',
    'original_hit_count',
    'total_asos_tested',
    'most_popular_canonical',
    'popular_hit_count'
]
print(present_originals[recoverable_cols].to_string(index=False))

--- MISSING ORIGINALS (13 groups) ---
These did not map to the original name at all (or missed the 50% threshold):

original_target_gene  original_hit_count  total_asos_tested most_popular_canonical  popular_hit_count match_rate
               BRCA2                   0                 37                N4BP2L2                 35      35/37
               MAPK6                   0                 37                 MAPK12                 24      24/37
                NAIP                   3                 76                   UNKL                 68      68/76
             NFKBIL1                   1                 77                  TONSL                 56      56/77
                NRF1                   0                 78                   NKRF                 74      74/78
               PTPRJ                   0                 71                  PTPRF                 59      59/71
               RASD1                   0                 77                  RASD2           

In [5]:
import pandas as pd

# The complete mapping dictionary for the 24 mismatched groups
canonical_mapping = {
    # Original is wrong (compilation error)
    "SFRS10": "TRA2B",        # Outdated alias
    "SENP2":  "SUMO2",        # Patent targeted SUMO2 but compiler confused it with enzyme that cuts it, SENP2
    "SRC":    "CSK",          # Patent targeted C-SRC tyrosine kinase which is CSK but GPT stopped at SRC
    "NRF1":   "NKRF",         # Patent targeted "NRF" (NF-kappa-B-repressing factor, now NKRF) but GPT confused with NRF1
    "BRCA2":  "N4BP2L2",      # Patent targeted BRCA2 region transcription unit CG005 which later renamed to N4BP2L2, GPT ignored the later fact.
    "MAPK6":  "MAPK12",       # Patent targeted ERK6, historic name for MAPK12, but compiler confused with ERK3 which was mentioned in the patent
    "NFKBIL1":"TONSL",        # Patent targeted IKB-R whihc is the old name for NFKBIL2 and officially TONSL, GPT guessed wrong
    "PTPRJ": "PTPRF",         # Patent target LAR which is alias for PTPRF, GPT mixed PTP receptor with PTPRJ
    "THRAP3": "ZNHIT3",        # Patent targeted TRIP3, which is an alias for ZNHIT3, but mistakenly transformed TRIP3 to THRAP3.
    # Original is wrong (patent / mislabel)
    "UBE3A":  "SNHG14",       # Patent *discusses* UBE3A regulation, but targets SNHG14 completely different gene UNKL
    "RASD1": "RASD2",         # Patent probably annotated incorrectly

    # The most popular disagrees with the original, we choose the original
    "ANGPT2": "ANGPT2",
    "ANGPTL3": "ANGPTL3",
    "BAG5": "BAG5",
    "FNTB": "FNTB",
    "IGF2": "IGF2",
    "IL18": "IL18",
    "MMP1": "MMP1",
    "MRPS16": "MRPS16",
    "PLN": "PLN",
    "S1PR2": "S1PR2",
    "STAT5B": "STAT5B",




    # Patent errors, unrecoverable
    "RHO": pd.NA,             # Patent targeting mutation, we don't mutate the genome.
    "NAIP": pd.NA,            # Patent probably contains an error (Old, genome wasn't annotated properly). ASOs target
}

In [7]:
from notebooks.consts import ORIGINAL_OLIGO_CSV_WITH_CANONICAL
from tauso.data.consts import CANONICAL_GENE

initial_count = len(df)

# 1. First, drop ASOs that don't even have a target_gene to begin with
df_no_empty_genes = df.dropna(subset=['target_gene']).copy()
dropped_empty = initial_count - len(df_no_empty_genes)
print(f"Dropped {dropped_empty} ASOs missing an initial 'target_gene'.")

# 2. Apply the map using .replace() to preserve our pd.NA traps
# We apply this to the intermediate dataframe we just created
df_no_empty_genes[CANONICAL_GENE] = df_no_empty_genes['target_gene'].replace(canonical_mapping)

# 3. Purge the unrecoverable data (the ones we mapped to pd.NA like RHO, NAIP)
df_clean = df_no_empty_genes.dropna(subset=[CANONICAL_GENE]).copy()
dropped_unrecoverable = len(df_no_empty_genes) - len(df_clean)

print(f"Dataset cleaned! Purged {dropped_unrecoverable} ASOs due to unrecoverable errors (e.g., RHO, NAIP).")
print(f"Final dataset contains {len(df_clean)} ASOs ready for TAUSO.")

Dropped 5282 ASOs missing an initial 'target_gene'.
Dataset cleaned! Purged 151 ASOs due to unrecoverable errors (e.g., RHO, NAIP).
Final dataset contains 173191 ASOs ready for TAUSO.


In [8]:
# 4. Save to file
df_clean.to_csv(ORIGINAL_OLIGO_CSV_WITH_CANONICAL, index=False)
print(f"Successfully saved to {ORIGINAL_OLIGO_CSV_WITH_CANONICAL}")

Successfully saved to /home/michael/career/tauso_article/tauso_source2/notebooks/data/OligoAI/raw_data/aso_inhibitions_with_canonical_gene.csv.gz


In [11]:
gene_stats[gene_stats['original_target_gene'] == 'DMPK']

,original_target_gene,total_asos_tested,most_popular_canonical,popular_hit_count,match_rate,all_hit_frequencies
75,DMPK,3304,DMPK,2636,2636/3304,"{'DM1-AS': 905, 'DMPK': 2636, 'RUSF1': 4, 'AC0..."
